In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [2]:
df = pd.read_csv("mental_health_dataset.csv")
df.head()

,age,gender,employment_status,work_environment,mental_health_history,seeks_treatment,stress_level,sleep_hours,physical_activity_days,depression_score,anxiety_score,social_support_score,productivity_score,mental_health_risk
0,56,Male,Employed,On-site,Yes,Yes,6,6.2,3,28,17,54,59.7,High
1,46,Female,Student,On-site,No,Yes,10,9.0,4,30,11,85,54.9,High
2,32,Female,Employed,On-site,Yes,No,7,7.7,2,24,7,62,61.3,Medium
3,60,Non-binary,Self-employed,On-site,No,No,4,4.5,4,6,0,95,97.0,Low
4,25,Female,Self-employed,On-site,Yes,Yes,3,5.4,0,24,12,70,69.0,High


# Problem formulation

The task is to predict `mental_health_risk` from demographic, behavioral and mental-health-related features.  
The methodological question studied here is not only whether logistic regression performs well, but whether a **custom numerical outlier detection rule** improves the classifier by removing potentially abnormal training samples.

So the problem is split into two parts:
- a **prediction problem**: classify `Low`, `Medium`, or `High` mental health risk,
- a **data treatment question**: determine whether removing numerically extreme points is useful or unnecessary for this dataset.


### Target and feature types

We separate:
- `y` = target to predict (`mental_health_risk`)
- `X` = all other features

We also separate columns:
- numerical columns (used for custom outlier detection)
- categorical columns (need one-hot encoding)


In [3]:
target_col = "mental_health_risk"
X = df.drop(columns=[target_col])
y = df[target_col]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()

num_cols, cat_cols


/tmp/ipykernel_45375/2235302591.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()


(['age',
  'stress_level',
  'sleep_hours',
  'physical_activity_days',
  'depression_score',
  'anxiety_score',
  'social_support_score',
  'productivity_score'],
 ['gender',
  'employment_status',
  'work_environment',
  'mental_health_history',
  'seeks_treatment'])

### Interpretation of feature separation

The features are divided into:
- **Numerical features** (age, stress, sleep, depression, anxiety, etc.), which represent measurable values and are used for outlier detection.
- **Categorical features** (gender, employment, work environment, mental health history), which describe context and are encoded later for classification.

Outlier detection is applied only to numerical features because abnormal values can only be defined on numbers.
Both feature types are then used together to predict the mental health risk.


### Preprocessing

The treatments used in this notebook are chosen to match the structure of the problem:
- numerical variables are standardized because logistic regression is scale-sensitive,
- categorical variables are one-hot encoded because the classifier needs numerical inputs,
- outlier detection is tested only on numerical variables because z-scores are defined for quantitative values.

The key point is that outlier detection is treated as a **candidate preprocessing step** whose usefulness must be verified, not assumed in advance.


In [5]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)


### Train/test split

We split data:
- training set: used to train
- test set: used only for final evaluation

We use stratify to keep the same class proportions in both splits.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape


((7500, 13), (2500, 13))

### Interpretation of the train / test split

After splitting the dataset, we obtain:
- 7,500 samples in the training set
- 2,500 samples in the test set

Each sample contains **13 features**.

This corresponds to a **75% / 25% split**, which is a common choice in machine learning:
- the training set is used to learn the model parameters
- the test set is used to evaluate performance on unseen data

The `stratify=y` option ensures that the proportions of
`Low`, `Medium`, and `High` mental health risk
are similar in both the training and test sets.

This makes the evaluation more reliable and avoids biased results.


### Baseline

We train logistic regression on the full training set.
This is the reference score.


In [7]:
Xt_train = preprocess.fit_transform(X_train)
Xt_test = preprocess.transform(X_test)

baseline_model = LogisticRegression(max_iter=5000)
baseline_model.fit(Xt_train, y_train)

baseline_pred = baseline_model.predict(Xt_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

baseline_acc


0.9988

### Interpretation of the baseline accuracy

The baseline classification accuracy is **0.9988**, which means that:
- about **99.88%** of the test samples are correctly classified
- the model makes very few mistakes on unseen data

This result is very high, showing that:
- the features are strongly correlated with the target variable
- logistic regression is sufficient to solve this classification problem
- the dataset is relatively clean and well-structured

This baseline result will be used as a **reference point**.
All models using outlier detection will be compared to this score
to see whether removing outliers improves or degrades performance.


### Custom outlier detection

We build our own outlier detector:
- take only standardized numerical features
- compute z-scores (how extreme values are)
- mark a sample as outlier if any numeric feature has |z| > threshold

This is a simple custom rule, not a library outlier model.


In [8]:
num_scaler = StandardScaler()
Xnum_train = num_scaler.fit_transform(X_train[num_cols])

threshold = 3.0
z = np.abs(Xnum_train)

inlier_mask = (z <= threshold).all(axis=1)

inlier_mask.mean(), inlier_mask.sum(), len(inlier_mask)


(np.float64(1.0), np.int64(7500), 7500)

### Interpretation of custom outlier detection result

With a z-score threshold of **3.0**, the custom outlier detection keeps:
- **100% of the training samples**
- **7,500 inliers out of 7,500 samples**

This means that, with this threshold, the outlier detector is **defined and evaluated**, but it is **not effectively applied** because it does not remove any sample.

So at threshold `3.0`, the situation is equivalent to training without outlier removal.  
This is an important methodological point: if no point is flagged, then the preprocessing rule exists, but it has no practical effect on the data used by the classifier.


### Train after removing outliers

We remove outliers only from the training set.
Then we train the same classifier again and test on the same test set.

If accuracy improves, it means outliers were adding noise.
If accuracy drops, it means we removed useful training data.


In [9]:
X_train_in = X_train.iloc[np.where(inlier_mask)[0]]
y_train_in = y_train.iloc[np.where(inlier_mask)[0]]

Xt_train_in = preprocess.fit_transform(X_train_in)
Xt_test = preprocess.transform(X_test)

model_inliers = LogisticRegression(max_iter=5000)
model_inliers.fit(Xt_train_in, y_train_in)

pred_inliers = model_inliers.predict(Xt_test)
acc_inliers = accuracy_score(y_test, pred_inliers)

baseline_acc, acc_inliers


(0.9988, 0.9988)

### Interpretation of accuracy after outlier removal

The accuracy after applying custom outlier detection is **0.9988**, which is **exactly the same as the baseline accuracy**.

This equality is expected because no training sample was removed at threshold `3.0`.  
So this experiment should not be interpreted as evidence that outlier removal helped; it should be interpreted as evidence that **the chosen threshold produced no actual treatment of the data**.

In other words:
- the detection rule was tested,
- but with this threshold it did not modify the training set,
- therefore the model trained afterwards is effectively the same as the baseline pipeline.


### Sweep threshold

The threshold controls how strict outlier detection is:
- low threshold -> removes more points
- high threshold -> removes fewer points

We test multiple thresholds and see how test accuracy changes.


In [10]:
threshold_list = [2.0, 2.5, 3.0, 3.5, 4.0]
acc_list = []
removed_ratio_list = []
kept_list = []
applied_list = []

for t in threshold_list:
    Xnum_train = num_scaler.fit_transform(X_train[num_cols])
    z = np.abs(Xnum_train)
    inlier_mask = (z <= t).all(axis=1)

    X_train_in = X_train.iloc[np.where(inlier_mask)[0]]
    y_train_in = y_train.iloc[np.where(inlier_mask)[0]]

    Xt_train_in = preprocess.fit_transform(X_train_in)
    Xt_test = preprocess.transform(X_test)

    clf = LogisticRegression(max_iter=5000)
    clf.fit(Xt_train_in, y_train_in)

    pred = clf.predict(Xt_test)
    acc = accuracy_score(y_test, pred)

    removed_ratio = 1 - inlier_mask.mean()
    acc_list.append(acc)
    kept_list.append(int(inlier_mask.sum()))
    removed_ratio_list.append(removed_ratio)
    applied_list.append(removed_ratio > 0)

results = pd.DataFrame(
    {
        'z_threshold': threshold_list,
        'kept_train_samples': kept_list,
        'removed_ratio': removed_ratio_list,
        'outlier_removal_applied': applied_list,
        'test_accuracy': acc_list
    }
)

results

,z_threshold,kept_train_samples,removed_ratio,outlier_removal_applied,test_accuracy
0,2.0,7078,0.056267,True,0.9980
1,2.5,7500,0.000000,False,0.9988
2,3.0,7500,0.000000,False,0.9988
3,3.5,7500,0.000000,False,0.9988
4,4.0,7500,0.000000,False,0.9988


### Interpretation of the threshold sweep results

The table distinguishes two different situations:
- thresholds for which outlier removal is **actually applied**,
- thresholds for which the detector finds no outlier, so the preprocessing is effectively a **no-op**.

- With threshold **2.0**:
  - about **5.6% of the training samples** are removed,
  - `outlier_removal_applied = True`,
  - test accuracy slightly decreases to **0.9980**.

  This is the only threshold in the sweep where the treatment is genuinely applied, and it slightly hurts performance.

- With thresholds **2.5 and higher**:
  - **no samples are removed**,
  - `outlier_removal_applied = False`,
  - test accuracy stays equal to the baseline.

These thresholds should therefore not be interpreted as successful outlier-removal settings; they simply reproduce the baseline because the detector does not modify the dataset.


### Selection of a meaningful threshold

There are two possible notions of "best":
- the best **test accuracy overall**,
- the best threshold among those where outlier removal is **actually applied**.

This distinction matters here because a threshold that removes nothing is methodologically equivalent to not applying outlier removal at all.


In [11]:
best_idx = int(np.argmax(results['test_accuracy'].values))
best_row = results.iloc[best_idx]

applied_results = results[results['outlier_removal_applied']]
if applied_results.empty:
    best_applied_row = None
else:
    best_applied_idx = int(np.argmax(applied_results['test_accuracy'].values))
    best_applied_row = applied_results.iloc[best_applied_idx]

best_row, best_applied_row

(z_threshold                   2.5
 kept_train_samples           7500
 removed_ratio                 0.0
 outlier_removal_applied     False
 test_accuracy              0.9988
 Name: 1, dtype: object,
 z_threshold                     2.0
 kept_train_samples             7078
 removed_ratio              0.056267
 outlier_removal_applied        True
 test_accuracy                 0.998
 Name: 0, dtype: object)

### Interpretation of the selected thresholds

The best row in terms of raw test accuracy is threshold **2.5**, with accuracy **0.9988**.  
But this threshold removes **no sample**, so it is not a meaningful outlier-removal result: it is just the baseline reproduced by an inactive detector.

Among the thresholds where outlier removal is **actually applied**, the only meaningful tested case is **2.0**:
- some samples are removed,
- the treatment is genuinely active,
- but the accuracy drops slightly to **0.9980**.

So the correct methodological conclusion is not "2.5 is the best outlier threshold".  
The correct conclusion is:
- thresholds `2.5+` do not really apply outlier removal,
- threshold `2.0` does apply it,
- and this genuine outlier-removal setting slightly degrades performance.

Therefore, for this dataset, the justified choice is to **not apply outlier removal**, because the only effective tested version of the treatment is not beneficial.
